# 🦜 Qwen Small Models Playground

Alibaba's **Qwen 3.5 Small** series (0.8B → 9B parameters) delivers serious capability on tiny hardware.
This notebook lets you **try every size** — text chat, streaming, and vision — with minimal setup.

| Model | Params | Vision | Best for |
|-------|--------|--------|----------|
| `qwen3.5:0.8b` | 800 M | ✅ | IoT, ultra-fast responses |
| `qwen3.5:1.5b` | 1.5 B | ✅ | Mobile, always-on assistants |
| `qwen3.5:3b`   | 3 B   | ✅ | Desktop apps, offline assistants |
| `qwen3.5:7b`   | 7 B   | ✅ | High-quality local inference |
| `qwen3.5:9b`   | 9 B   | ✅ | Best-in-class small model |
| `qwen3.5-plus` | cloud | ✅ | DashScope API, no GPU needed |

> 🦃 *Created by Crunch · [Copilotclaw/trainingdemo](https://github.com/Copilotclaw/trainingdemo)*


## 🛠️ Group 1 — Setup

Install dependencies and pick your backend.

In [ ]:
# ── Install dependencies ────────────────────────────────────────────────
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

pip_install("openai")
pip_install("Pillow")
pip_install("requests")

print("✅ Dependencies installed")


In [ ]:
# ── Choose your backend ─────────────────────────────────────────────────
#
#  BACKEND = "ollama"      → local Qwen via Ollama (free, private, offline)
#  BACKEND = "dashscope"   → cloud API (no GPU needed, needs API key)
#

BACKEND = "ollama"      # <── change me

# ── Ollama: pick your model size ─────────────────────────────────────────
#   Options: "qwen3.5:0.8b"  "qwen3.5:1.5b"  "qwen3.5:3b"
#            "qwen3.5:7b"    "qwen3.5:9b"
OLLAMA_MODEL = "qwen3.5:1.5b"

# ── DashScope API (get key at https://dashscope.aliyuncs.com) ────────────
DASHSCOPE_API_KEY  = "sk-..."   # paste your key here
DASHSCOPE_BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"
DASHSCOPE_MODEL    = "qwen3.5-plus"   # vision-capable cloud model

# ── Vision model? All Qwen 3.5 models support vision ─────────────────────
SUPPORTS_VISION = True

print(f"Backend: {BACKEND}")
print(f"Model  : {OLLAMA_MODEL if BACKEND == 'ollama' else DASHSCOPE_MODEL}")


## ⚙️ Group 2 — Client & Ollama Setup

Start the Ollama server (local only) and create a reusable chat client.

In [ ]:
# ── Start Ollama (skip if using DashScope) ──────────────────────────────
import shutil, time, subprocess

if BACKEND == "ollama":
    if not shutil.which("ollama"):
        print("Installing Ollama...")
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",
                       shell=True, check=True)
    else:
        print("Ollama already installed ✓")

    # Start server if not running
    try:
        import urllib.request
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        print("Ollama server already running ✓")
    except Exception:
        print("Starting Ollama server...")
        subprocess.Popen(["ollama", "serve"],
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(3)

    # Pull the model if needed
    print(f"Pulling {OLLAMA_MODEL} (first run may take a few minutes)...")
    subprocess.run(["ollama", "pull", OLLAMA_MODEL], check=True)
    print(f"✅ {OLLAMA_MODEL} ready")

else:
    print("Using DashScope — no local setup needed ✅")


In [ ]:
# ── Create OpenAI-compatible client ─────────────────────────────────────
from openai import OpenAI

if BACKEND == "ollama":
    client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    ACTIVE_MODEL = OLLAMA_MODEL
else:
    client = OpenAI(base_url=DASHSCOPE_BASE_URL, api_key=DASHSCOPE_API_KEY)
    ACTIVE_MODEL = DASHSCOPE_MODEL

# ── Helper: simple chat ──────────────────────────────────────────────────
def chat(prompt: str, system: str = "You are a helpful assistant.",
         model: str = None, max_tokens: int = 512) -> str:
    response = client.chat.completions.create(
        model=model or ACTIVE_MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt},
        ],
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

print(f"✅ Client ready — using model: {ACTIVE_MODEL}")


## 💬 Group 3 — Text Chat Playground

Just talk to your model. Change `MY_PROMPT` to anything you like.

In [ ]:
# ── Try a simple prompt ─────────────────────────────────────────────────
MY_PROMPT = "Explain the difference between a 1B and a 9B language model in 3 sentences."

reply = chat(MY_PROMPT)
print(f"🤖 [{ACTIVE_MODEL}]\n")
print(reply)


In [ ]:
# ── Change the system role ──────────────────────────────────────────────
# Try giving the model a different persona

PERSONA = "You are a pirate who explains technology with nautical metaphors."
QUESTION = "What is a neural network?"

reply = chat(QUESTION, system=PERSONA)
print(f"🤖 [{ACTIVE_MODEL} as pirate]\n")
print(reply)


## ⚡ Group 4 — Streaming

Watch tokens arrive in real-time — great for longer responses.

In [ ]:
# ── Streaming chat ──────────────────────────────────────────────────────
import sys

STREAMING_PROMPT = "Write a short poem about running AI models on tiny devices."

print(f"🤖 [{ACTIVE_MODEL}] (streaming)\n")
stream = client.chat.completions.create(
    model=ACTIVE_MODEL,
    messages=[{"role": "user", "content": STREAMING_PROMPT}],
    stream=True,
    max_tokens=200,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)

print("\n\n✅ Stream complete")


## 🏆 Group 5 — Model Showdown

Run the **same prompt across multiple model sizes** and compare the responses side by side.

> ⚠️ Ollama: this will pull multiple models — give it a few minutes on first run.  
> DashScope: only one cloud model is available; this section is most useful with Ollama.

In [ ]:
# ── Pick which models to compare ────────────────────────────────────────
if BACKEND == "ollama":
    COMPARE_MODELS = ["qwen3.5:0.8b", "qwen3.5:1.5b", "qwen3.5:3b"]
else:
    # DashScope — we compare with a slightly different prompt each time
    COMPARE_MODELS = [DASHSCOPE_MODEL]   # only one cloud model; feel free to add others

SHOWDOWN_PROMPT = "In exactly two sentences: what is quantum computing?"

print("Pulling comparison models (Ollama only)...")
for m in COMPARE_MODELS:
    if BACKEND == "ollama":
        subprocess.run(["ollama", "pull", m], check=True, capture_output=True)
        print(f"  ✓ {m}")

print("\nRunning showdown...\n" + "="*60)
for m in COMPARE_MODELS:
    print(f"\n📦 {m}")
    print("-" * 40)
    try:
        r = chat(SHOWDOWN_PROMPT, model=m, max_tokens=150)
        print(r)
    except Exception as e:
        print(f"  ⚠️ Error: {e}")


## 👁️ Group 6 — Vision Playground

All Qwen 3.5 models include **native vision** — text and images are handled by the same model.  
Send an image URL and ask questions about it.

In [ ]:
# ── Vision: describe an image ───────────────────────────────────────────
import requests
from IPython.display import Image, display

if not SUPPORTS_VISION:
    print("⚠️ Your selected model doesn't support vision — switch to a Qwen3.5 model.")
else:
    # A simple public test image (Wikipedia Commons — Alibaba HQ)
    IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/9/9e/Alibaba_Group_logo.svg/320px-Alibaba_Group_logo.svg.png"

    print("Image:")
    display(Image(url=IMAGE_URL, width=300))

    print("\nSending image to model...\n")

    response = client.chat.completions.create(
        model=ACTIVE_MODEL,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": IMAGE_URL}},
                {"type": "text",      "text": "What do you see in this image? One short paragraph."},
            ],
        }],
        max_tokens=200,
    )

    print(f"🤖 [{ACTIVE_MODEL}]\n")
    print(response.choices[0].message.content)


In [ ]:
# ── Vision: visual Q&A ──────────────────────────────────────────────────
# Swap in any image URL and ask your own question

MY_IMAGE_URL  = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png"
MY_QUESTION   = "What objects are in the image? List them."

if SUPPORTS_VISION:
    display(Image(url=MY_IMAGE_URL, width=300))

    response = client.chat.completions.create(
        model=ACTIVE_MODEL,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": MY_IMAGE_URL}},
                {"type": "text",      "text": MY_QUESTION},
            ],
        }],
        max_tokens=300,
    )

    print(f"\n❓ {MY_QUESTION}\n")
    print(f"🤖 {response.choices[0].message.content}")


## 📁 Group 7 — Vision with Local File

Send a **local image file** (e.g., a screenshot or photo from your machine) to the model.

In [ ]:
# ── Upload a local image and describe it ────────────────────────────────
import base64
from pathlib import Path

if SUPPORTS_VISION:
    # Option A: upload via Colab file picker
    try:
        from google.colab import files
        print("Choose an image file to upload...")
        uploaded = files.upload()
        filename = next(iter(uploaded))
        image_bytes = uploaded[filename]
    except ImportError:
        # Local Jupyter: load from disk
        filename = "test.jpg"  # <── change to your image path
        image_bytes = Path(filename).read_bytes()

    # Base64-encode the image
    b64 = base64.b64encode(image_bytes).decode()
    ext = Path(filename).suffix.lstrip(".") or "jpeg"
    data_url = f"data:image/{ext};base64,{b64}"

    response = client.chat.completions.create(
        model=ACTIVE_MODEL,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": data_url}},
                {"type": "text",      "text": "Describe this image in detail."},
            ],
        }],
        max_tokens=400,
    )
    print(f"\n🤖 [{ACTIVE_MODEL}]\n")
    print(response.choices[0].message.content)
